# 04 — Visible assortment

This notebook presents the **visible assortment** of the validated pricing
population: how many distinct products each approved canonical location
returned at each scheduled capture, which products were added or removed
between consecutive captures, how stable consecutive product sets were, which
intervals show an observed drop, and whether assortment changes coincided
with price changes in the same interval.

It is a thin presentation layer. Every product set, interval, ratio, price
outcome, reconciliation and privacy rule comes from `ql2_sixt_canada_analysis`
(`run_assortment_presentation` → `run_pricing_pipeline` → price-change engine
→ visible-assortment engine → presentation). It does not depend on earlier
notebooks or kernel state.

The results are descriptive. Observed drops are review candidates, not proven
anomalies, and nothing here proves a collection failure or a supplier
withdrawal.

## Question and scope

Data-plan Section 5, *Visible assortment*:

- count returned products by location and capture;
- identify additions and removals;
- calculate consecutive-capture retention;
- calculate Jaccard similarity;
- identify unusual assortment drops (observed drop metrics only; see below);
- check whether assortment changes coincide with price changes;
- produce an assortment timeline.

The question is what the collection *returned*, which is not necessarily
what the supplier offered.

## Definitions

The definitions come from the visible-assortment contract and are not
redefined here.

- **Population:** pricing-eligible canonical offers. Vancouver Downtown and
  Thurlow are one canonical location.
- **Capture:** the authority-backed scheduled capture period, on the schedule
  grid. An eligible capture with no offers counts zero products. Governed
  exclusions and missing captures have no count.
- **Product:** the approved product identity (name, type, transmission, seats,
  bags). Price, currency and price basis are not part of it, and several
  offers of one product count once.
- **Valid interval:** two eligible captures exactly one hour apart with the
  same contributing streams and the same rental search context. Every other
  adjacent pair is a typed break, and nothing is compared across it.
- **Formulas:** for previous set P and current set C:
  - retention = |P∩C| / |P|, with the previous set as denominator;
  - Jaccard = |P∩C| / |P∪C|;
  - net change = |C| − |P|;
  - absolute drop = max(|P| − |C|, 0);
  - drop rate = absolute drop / |P|.
- **Zero denominators:** an empty previous set (or an empty union) gives an
  explicit `zero_denominator` status instead of a value.
- **Unusual drops:** the unusual-drop policy is unavailable (proposed, not
  approved). `unusual_drop` stays empty, and observed drops are review
  candidates only.
- **Price coincidence:** an increase or decrease candidate in the same
  canonical location and the same scheduled interval. Increases and decreases
  are kept separate, and price coincidence is not causation.

## Confidentiality

All QL2 data is proprietary. This notebook shows only sanitized aggregate
tables and the aggregate timeline figure. It never shows product identities,
prices, rental dates, raw identifiers, source labels, file names or paths.

Timeline persistence is not approved, so nothing is written. Tables and the
figure exist only in memory during local execution. Clear all outputs before
committing (Edit ▸ Clear Outputs of All Cells).

In [ ]:
from IPython.display import Image, display

from ql2_sixt_canada_analysis import (
    assortment_timeline_png,
    build_assortment_narrative,
    run_assortment_presentation,
)
from ql2_sixt_canada_analysis.paths import resolve_raw_data_dir

# Raw-data directory: the centralized default unless the documented environment
# override is set. Set RAW_DIR_OVERRIDE only for a deliberate local experiment.
RAW_DIR_OVERRIDE = None

raw_dir = resolve_raw_data_dir(RAW_DIR_OVERRIDE)
print("Raw-data directory selected.")

## Readiness and evidence

One call runs the pricing pipeline exactly once and passes that same result
through the price-change engine, the visible-assortment engine and every
reconciliation gate. If anything is blocked, only the blocker categories are
shown, and no table, figure or finding is produced.

In [ ]:
presentation = run_assortment_presentation(raw_dir)
ready = presentation.completed
print("Visible-assortment presentation status:", presentation.report.status.value)
if ready:
    tables = presentation.tables
    print("All presentation reconciliation checks passed:", presentation.report.reconciled)
    print("Unusual-drop policy status:", presentation.report.anomaly_policy_status.value)
    print("Timeline persistence approved:", presentation.report.persistence_approved)
else:
    print("Blocked - no tables, figure or findings are produced.")
    print("Blockers:", ", ".join(b.value for b in presentation.report.blockers))
    if presentation.report.upstream_blockers:
        print("Upstream blockers:", ", ".join(presentation.report.upstream_blockers))

In [ ]:
if ready:
    display(tables.reconciliation_summary)
else:
    print("Skipped: the presentation is blocked.")

## Assortment timeline

There is one row per approved canonical location and scheduled capture. This
includes seed captures, assessed intervals, eligible captures with zero
products, governed exclusions, missing captures and typed breaks, each with
its explicit status. In the figure, each location's line breaks at every
interval break. Shaded captures have no count (they are not zero). Triangles
mark observed drops, and rings mark observed drops with a same-interval price
increase. The figure is rendered in memory from the reconciled table.

In [ ]:
if ready:
    display(tables.assortment_timeline)
    display(Image(data=assortment_timeline_png(tables)))
else:
    print("Skipped: the presentation is blocked.")

## Additions, removals, retention, and Jaccard

One row per approved canonical location. Every ratio statistic carries its
contributor count: retention and Jaccard medians use only assessed intervals
with a defined value. Zero denominators are counted separately, and seed
captures and captures after a break are in no denominator. Product identities
are never shown.

In [ ]:
if ready:
    display(tables.location_summary)
else:
    print("Skipped: the presentation is blocked.")

## Observed-drop review

Each row is an assessed interval with an absolute drop above zero. The rows
are ordered by largest drop first for review only, and that order is not a
statistical ranking. The drop pattern separates three cases: a net
contraction, a complete turnover and an empty current capture.

The second table groups observed drops by exact scheduled period. One
location is *isolated in this extract*, and several are *simultaneous in this
extract*. Simultaneity is not evidence of a common cause. These are review
candidates, not proven anomalies, and the unusual-drop policy is
unavailable.

In [ ]:
if ready:
    display(tables.observed_drop_review)
    display(tables.cross_location_drops)
else:
    print("Skipped: the presentation is blocked.")

## Price-change coincidence

These counts cover assessed intervals at each location, with increases and
decreases kept separate. A coincidence requires the same canonical location
and the same previous and current scheduled periods. `unchanged`, `appeared`,
`disappeared` and `ambiguous` price outcomes are never price changes. Shares
are shown only when their denominator is non-zero. Price coincidence is not
causation.

In [ ]:
if ready:
    display(tables.price_coincidence_summary)
else:
    print("Skipped: the presentation is blocked.")

## Interpretation and limitations

The narrative below is generated deterministically from the validated
aggregate tables. When the presentation is blocked, it states only the
blocker categories.

- The collection spans roughly 90 hours, too short for seasonal or long-term
  baselines.
- Visible assortment is what the collection returned, not proof of supplier
  availability.
- Operational and source corroboration would be required for any causal
  claim.

In [ ]:
narrative = build_assortment_narrative(presentation)
print(narrative.text)

## Monitoring and actionability

No monitoring rule or threshold is established here. A future assortment drop
check needs four approved decisions from the business owner: the method, the
minimum history, the grouping and the thresholds. Until then, the
observed-drop review is a list of review candidates. The tables suggest
questions for the collection owner (was the capture complete?) and for the
revenue-management team (did the visible range change?). They answer neither.

The data-plan Section 6 control catalog is in `05_monitoring_actionability.ipynb`. It keeps this
check candidate-only until such a policy is approved.

## Data-plan reconciliation

- Count returned products by location and capture: presented in the aggregate timeline.
- Identify additions and removals: presented for every valid consecutive interval and summarized by location.
- Calculate consecutive-capture retention: presented with the previous set as denominator and explicit zero-denominator status.
- Calculate Jaccard similarity: presented as intersection divided by union with explicit empty-union status.
- Identify unusual assortment drops: observed drop metrics and review candidates are presented; statistical unusualness and alerting remain unavailable pending approved policy.
- Check whether assortment changes coincide with price changes: presented by exact canonical location and interval, with increases and decreases separate and no causal claim.
- Produce an assortment timeline: implemented as a validated in-memory aggregate table and an in-memory visualization. Persistence remains unavailable until approved.

## Clear all outputs before committing

Use Edit ▸ Clear Outputs of All Cells before committing. `tests/test_notebooks.py`
fails if a committed notebook has outputs or execution counts.

In [ ]:
print("No files written.")